[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/41_opd_loss.ipynb)

# 🔴 Hard: OPD Loss

Implement the **On-Policy Distillation (OPD)** loss used to distill one or more teacher policies into a student policy on trajectories sampled from the student.

For each token position, OPD minimizes a weighted reverse KL from the student distribution to each teacher distribution:

$$\mathcal{L}_{\text{OPD}} = \sum_j w_j\,D_{\text{KL}}\big(\pi_\theta(\cdot \mid x)\;||\;\pi_{E_j}(\cdot \mid x)\big)$$

where

$$D_{\text{KL}}(p||q) = \sum_v p(v)\,[\log p(v) - \log q(v)].$$

### Signature
```python
from torch import Tensor

def opd_loss(student_logits: Tensor,
             teacher_logits: Tensor,
             teacher_weights: Tensor | None = None,
             mask: Tensor | None = None,
             temperature: float = 1.0) -> Tensor:
    """OPD reverse-KL distillation loss.

    student_logits: (..., V) logits from the student policy
    teacher_logits: (..., V) for one teacher, or (T, ..., V) for T teachers
    teacher_weights: optional (T,) weights, normalized internally
    mask: optional (...) token mask, where 1 = include and 0 = ignore
    temperature: softmax temperature used for distillation
    returns: scalar loss (Tensor)
    """
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [14]:
def opd_loss(student_logits: Tensor,
             teacher_logits: Tensor,
             teacher_weights: Tensor | None = None,
             mask: Tensor | None = None,
             temperature: float = 1.0) -> Tensor:
    """OPD reverse-KL distillation loss.

    student_logits: (..., V) logits from the student policy
    teacher_logits: (..., V) for one teacher, or (T, ..., V) for T teachers
    teacher_weights: optional (T,) weights, normalized internally
    mask: optional (...) token mask, where 1 = include and 0 = ignore
    temperature: softmax temperature used for distillation
    returns: scalar loss (Tensor)
    """

    if temperature <= 0:
        raise ValueError("temperature must be positive")

    log_s = F.log_softmax(
        student_logits.float() / temperature, dim=-1
    )  # (N, V) or (B, N, V)

    log_t = F.log_softmax(
        teacher_logits.detach().float() / temperature, dim=-1
    ) # (T, N, V) or (T, B, N, V)

    # add teacher axis even in the case of a single teacher
    if log_t.ndim == log_s.ndim:
        log_t = log_t.unsqueeze(0)  # (1, N, V) or (1, B, N, V)

    n_teachers = log_t.shape[0]

    if teacher_weights is None:
        weights = torch.ones((n_teachers,)) #(T, )
    else:
        weights = teacher_weights.detach() #(T, )

    kl = torch.sum(log_s.exp() * (log_s - log_t), dim = -1) #(T, N) or (T, B, N)
    #normalize weights
    weights = (weights / weights.sum()).reshape((n_teachers, ) + (1, ) * (kl.ndim - 1)) #(T, 1) or (T, 1, 1)

    loss = torch.sum(weights * kl[torch.arange(n_teachers), :], dim = 0) * temperature ** 2 #sum over teachers --> (N, ) or (B, N), check theroy for '* temperature ** 2' term
    if mask is None:
        return loss.mean() #()

    return (loss * mask).sum() / (mask.sum()) #mean over all included tokens --> ()

In [15]:
# 🧪 Debug
student_logits = torch.tensor([[[2.0, 0.0, -1.0], [0.5, 1.0, -0.5]]])
teacher_logits = torch.tensor([[[1.0, 1.5, -0.5], [0.0, 2.0, -1.0]]])
print('Loss:', opd_loss(student_logits, teacher_logits).item())

Loss: 0.39124953746795654


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('opd_loss')


🧪 Testing: OPD (On-Policy Distillation) Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/7] Basic shape & type (2.0ms)
  ✅ [2/7] Zero when student matches teacher (3.5ms)
  ✅ [3/7] Numeric check vs single-teacher reverse KL (0.8ms)
  ✅ [4/7] Multi-teacher weighted reverse KL (0.8ms)
  ✅ [5/7] Mask ignores padded tokens (0.7ms)
  ✅ [6/7] Gradient flows only through student logits (0.7ms)
  ✅ [7/7] Temperature scaling (0.7ms)
──────────────────────────────────────────────────
  🎉 All 7 tests passed! (9.2ms total)
  Progress saved. Run status() to see your dashboard.

